In [4]:
import pandas as pd
import geopandas as gpd
import shapely
from shapely import wkt
from shapely.validation import make_valid
from shapely.geometry import Polygon, MultiPolygon
from shapely.strtree import STRtree
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import warnings
from tqdm import tqdm
import time
import os
import glob

In [5]:
df_a = pd.read_csv('cup_it_example_src_A_cleaned.csv', index_col=0)
df_b = pd.read_csv('cup_it_example_src_B_cleaned.csv', index_col=0)

/tmp/ipykernel_6726/3017995947.py:2: DtypeWarning: Columns (0: housing) have mixed types. Specify dtype option on import or set low_memory=False.
  df_b = pd.read_csv('cup_it_example_src_B_cleaned.csv', index_col=0)


In [7]:
df_a[~df_a['gkh_address'].isna()].head()

,title,tags,geometry,area_sq_m,gkh_address,gkh_floor_count_min,gkh_floor_count_max,geometry_original,area_m2,component,is_isolated,centroid,union
13,NaN,['жилое здание'],MULTIPOLYGON (((679554.600781885 6654108.08005...,75.89210,"г. Санкт-Петербург, ул. Яхтенная, д. 3, к. 2",17.0,25.0,"MULTIPOLYGON (((30.2185 59.98499, 30.21845 59....",75.878480,12,False,POINT (679538.5451098043 6654100.861703715),"POLYGON ((679552.2026421655 6654116.71516149, ..."
18,NaN,['жилое здание'],MULTIPOLYGON (((679896.9633155947 6654296.5441...,732.11580,"г. Санкт-Петербург, ул. Савушкина, д. 121, к. 1",16.0,17.0,"MULTIPOLYGON (((30.22479 59.98653, 30.22454 59...",732.317790,16,False,POINT (679903.7357410978 6654339.870268532),"POLYGON ((679907.9586557933 6654325.395708679,..."
20,NaN,['жилое здание'],MULTIPOLYGON (((681054.6252438403 6648008.4871...,1057.69400,"г. Санкт-Петербург, ул. Опочинина, д. 15/18, с...",5.0,5.0,"MULTIPOLYGON (((30.23999 59.92965, 30.24001 59...",1057.847437,18,False,POINT (681119.4231284221 6647871.06475608),MULTIPOLYGON (((681102.3895776458 6647879.2043...
45,NaN,['жилое здание'],MULTIPOLYGON (((683042.4870645727 6649798.6076...,930.03400,"г. Санкт-Петербург, линия. 6-я В.О., д. 37, ли...",5.0,5.0,"MULTIPOLYGON (((30.27708 59.94482, 30.27712 59...",930.349678,41,False,POINT (683069.615711096 6649799.368241046),MULTIPOLYGON (((683098.8570682309 6649779.0919...
59,NaN,['жилое здание'],MULTIPOLYGON (((684007.1094139321 6647441.5029...,691.32925,"г. Санкт-Петербург, пр-кт. Лермонтовский, д. 8...",6.0,7.0,"MULTIPOLYGON (((30.29222 59.92326, 30.29254 59...",691.677278,54,False,POINT (684070.5175731728 6647482.676432438),MULTIPOLYGON (((684086.6247473876 6647509.0458...


In [8]:
df_a['gkh_address'].isna().value_counts()

gkh_address
True     117275
False     17085
Name: count, dtype: int64

In [9]:
df_b.columns

Index(['subject', 'district', 'type', 'locality', 'type_street', 'name_street',
       'number', 'letter', 'fraction', 'housing', 'building',
       'purpose_of_building', 'stairs', 'avg_floor_height', 'height', 'wkt',
       'id', 'stairs_numeric', 'avg_floor_height_numeric', 'height_calculated',
       'height_numeric', 'geometry', 'area_m2', 'component', 'is_isolated',
       'centroid', 'union'],
      dtype='str')

In [15]:
required_to_load = []

def lacks_location_info(df_b_row):
    return pd.isna(df_b_row['name_street']) or pd.isna(df_b_row['number']) or pd.isna(df_b_row['locality'])

def serialize_centroid_of_wkt(wkt):
    crd = shapely.wkt.loads(wkt).centroid
    point = (crd.x, crd.y)
    return str(point)

for i, row in df_a.iterrows():
    if pd.isna(row['gkh_address']):
        required_to_load.append(serialize_centroid_of_wkt(row['geometry_original']))

for i, row in df_b.iterrows():
    if lacks_location_info(row):
        required_to_load.append(serialize_centroid_of_wkt(row['wkt']))

len(required_to_load)

132578

In [16]:
with open('required_to_load.txt', 'w') as f:
    f.write('\n'.join(required_to_load))